In [8]:
import os
import tqdm
import numpy as np
import pandas as pd
from tushare import pro_api, set_token
from datetime import datetime, timedelta
from datatools import get_price
import time

In [6]:
set_token('b549c6e18f71105519447d872727cc2b7f0022f071c9eb27d0256ebb')
pro = pro_api()

In [ ]:
#trade_cal = pro.trade_cal()
#trade_cal.to_csv('data/trade_cal.csv', index=False)
trade_cal = pd.read_csv('data/trade_cal.csv', dtype={'cal_date': str})
data_cal = trade_cal[(trade_cal['cal_date']>='20140101')&(trade_cal['cal_date']<'20260101')]

In [ ]:
data_path = 'data/daily_K/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily = pro.daily(trade_date=d['cal_date'])
        adj_factor = pro.adj_factor(trade_date=d['cal_date'])
        price = pd.merge(daily, adj_factor)
        price = price.drop(columns=['pre_close', 'change', 'pct_chg']).sort_values('ts_code').reset_index(drop=True)
        price['trade_date'] = pd.to_datetime(price['trade_date'])
        price.to_feather(os.path.join(data_path, f'stock-{d["cal_date"]}.ftr'))
    else:
        continue

In [ ]:
data_path = 'data/daily_basic/'
for _, d in tqdm.tqdm(data_cal.iterrows()):
    if d.is_open == 1:
        daily_basic = pro.daily_basic(trade_date=d['cal_date']).sort_values('ts_code').reset_index(drop=True)
        daily_basic['trade_date'] = pd.to_datetime(daily_basic['trade_date'])
        daily_basic.to_feather(os.path.join(data_path, f'basic-{d["cal_date"]}.ftr'))
    else:
        continue

In [ ]:
#industry = pro.index_member_all()
#industry.to_csv('data/industry.csv', index=False)
#allstock = pro.stock_basic()
#allstock.to_csv('data/allstock.csv', index=False)

In [ ]:
allstock = pd.read_csv('data/allstock.csv')
allstock_list = allstock.ts_code.tolist()
fins = []

In [15]:
len(fins)

4200

In [ ]:
batch_size = 200
with tqdm.tqdm(total=len(allstock_list), desc="处理股票", unit="stock") as pbar:
    for i in range(0, len(allstock_list), batch_size):
        batch = allstock_list[i:i + batch_size]
        
        for d in batch:
            tmp = pro.fina_indicator(ts_code=d, start_date='20130930', end_date='20260101').drop_duplicates(subset=['ts_code', 'end_date'], keep='first')
            fins.append(tmp)
            pbar.update(1)
        
        # 每批次结束后休息
        if i + batch_size < len(allstock_list):
            time.sleep(35)

处理股票: 100%|██████████| 1286/1286 [06:27<00:00,  3.32stock/s] 


In [17]:
df_fins = pd.concat(fins)
for d in df_fins.end_date.unique():
    df_fins[df_fins['end_date'] == d].to_feather(os.path.join('data/finance/', f'fina-{d}.ftr'))

C:\Users\yiluuu\AppData\Local\Temp\ipykernel_2064\1441635360.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_fins = pd.concat(fins)
